In [0]:
import time
from pyspark.sql import functions as F

In [0]:
%run ../00_utils/common

In [0]:
BATCH = batch_id()

## dim_productos

Linaje:
- art_id <- silver.mstr_articulos.art_id
- margen_estimado <- (precio_lista - precio_lista*0.6) / precio_lista
  - asume COGS = 60% del precio lista por categoría
- jerarquia_cat <- campos planos id_categ_n1/n2/n3 de mstr_articulos

In [0]:
def build_dim_productos():
      print(" === Inicio dim_productos ===")
      t0 = time.time()
      arts = spark.table(f"{CATALOG}.silver.mstr_articulos")
      provs = spark.table(f"{CATALOG}.silver.mstr_proveedores")

      # Margen estimado por categoría (COGS supuesto 60%)
      cogs_pct = {"CAT4": 0.55, "CAT1": 0.65, "CAT2": 0.62, "CAT3": 0.63, "CAT5": 0.60, "CAT6": 0.61}
      cogs_map = F.create_map([F.lit(k) for pair in cogs_pct.items() for k in pair])

      df = (arts.join(provs.select("id_proveedor", "razon_social", "pais_origen", "tiempo_repo_dias", "calificacion_calidad"), on="id_proveedor", how="left")
            .withColumn("cogs_pct",
                        F.coalesce(cogs_map[F.col("id_categ_n1")], F.lit(0.62)))
            .withColumn("margen_estimado",
                        F.round((1 - F.col("cogs_pct")) * 100, 2))
            .drop("cogs_pct", "_ingested_at", "_source_system", "_batch_id", "_ingest_year", "_ingest_month", "_ingest_day")
            .withColumn("_batch_id", F.lit(BATCH)))

      upsert(df, "gold.dim_productos", ["art_id"])
      log_run("gold", "dim_productos", df.count(), 0, time.time() - t0, batch=BATCH)
      print(f"  [DONE] gold.dim_productos")

## dim_tiendas
Linaje:
- zona_distribucion <- regla:   
  - Bogotá/Medellín/Cali -> CD_BOGOTA,
  - CDMX/Guadalajara/Monterrey -> CD_MEXICO,
  - Santiago/Valparaíso -> CD_SANTIAGO, 
  - resto -> CD_LOCAL

In [0]:
def build_dim_tiendas():
    print(" === Inicio dim_tiendas ===")
    t0 = time.time()
    df = spark.table(f"{CATALOG}.silver.mstr_tiendas")

    zona_expr = (
        F.when(F.col("id_ciudad").isin("Bogotá","Medellín","Cali","Barranquilla","Cartagena"), F.lit("CD_BOGOTA"))
         .when(F.col("id_ciudad").isin("Ciudad de Mexico","Guadalajara","Monterrey","Puebla","Tijuana"), F.lit("CD_MEXICO"))
         .when(F.col("id_ciudad").isin("Santiago","Valparaíso","Concepción","La Serena","Antofagasta"), F.lit("CD_SANTIAGO"))
         .otherwise(F.lit("CD_LOCAL"))
    )

    df = (df.withColumn("zona_distribucion", zona_expr)
          .drop("_ingested_at", "_source_system", "_batch_id", "_ingest_year", "_ingest_month", "_ingest_day")
          .withColumn("_batch_id", F.lit(BATCH)))

    upsert(df, "gold.dim_tiendas", ["id_tienda"])
    log_run("gold", "dim_tiendas", df.count(), 0, time.time() - t0, batch=BATCH)
    print(f"  [DONE] gold.dim_tiendas")

## dim_clientes
Linaje:
- antiguedad_dias <- datediff(current_date, fec_registro)
- id_miembro_hash <- heredado de silver (PII enmascarado)

In [0]:
def build_dim_clientes():
    print(" === Inicio dim_clientes ===")
    t0 = time.time()
    df = spark.table(f"{CATALOG}.silver.crm_miembros")

    df = (df.withColumn("antiguedad_dias", F.datediff(F.current_date(), F.col("fec_registro")))
            .drop("_ingested_at", "_source_system", "_batch_id", "_ingest_year", "_ingest_month", "_ingest_day")
            .withColumn("_batch_id", F.lit(BATCH)))

    upsert(df, "gold.dim_clientes", ["id_miembro"])
    log_run("gold", "dim_clientes", df.count(), 0, time.time() - t0, batch=BATCH)
    print(f"  [DONE] gold.dim_clientes")

## Execution

In [0]:
print(f"=== Gold Dimensions | batch={BATCH} ===")
try: build_dim_productos()
except Exception as e: print(f"  [ERROR] dim_productos: {e}")
try: build_dim_tiendas()
except Exception as e: print(f"  [ERROR] dim_tiendas: {e}")
try: build_dim_clientes()
except Exception as e: print(f"  [ERROR] dim_clientes: {e}")
print("\n[DONE] Gold dimensions completado.")